# Recurrent Neural Network (RNN) with TensorFlow & Keras

This notebook demonstrates RNN implementations for time series prediction using:
- **SimpleRNN** — basic recurrent layer
- **LSTM** (Long Short-Term Memory) — handles long-range dependencies
- **GRU** (Gated Recurrent Unit) — efficient alternative to LSTM

**Task:** Predict a noisy sine wave (univariate time series).

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import datetime
from tensorflow import keras

print("TensorFlow version:", tf.__version__)

## 1. Generate Time Series Data

We generate a noisy sine wave and split it into overlapping windows.
Each window of `n_steps` time steps is used to predict the next value.

In [ ]:
# --- Parameters ---
N_SAMPLES = 1200
N_STEPS   = 30      # look-back window
TRAIN_SIZE = 1000

# Generate noisy sine wave
np.random.seed(42)
time = np.linspace(0, 8 * np.pi, N_SAMPLES)
series = np.sin(time) + 0.15 * np.random.randn(N_SAMPLES)

plt.figure(figsize=(12, 3))
plt.plot(time[:200], series[:200], linewidth=0.8)
plt.title("Noisy Sine Wave (first 200 points)")
plt.xlabel("Time")
plt.ylabel("Value")
plt.tight_layout()
plt.show()

In [ ]:
def make_windows(series, n_steps):
    """Slide a window across the series to create (X, y) pairs."""
    X, y = [], []
    for i in range(len(series) - n_steps):
        X.append(series[i : i + n_steps])
        y.append(series[i + n_steps])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_all, y_all = make_windows(series, N_STEPS)

# Reshape to (samples, timesteps, features)
X_all = X_all[..., np.newaxis]

X_train, y_train = X_all[:TRAIN_SIZE], y_all[:TRAIN_SIZE]
X_test,  y_test  = X_all[TRAIN_SIZE:], y_all[TRAIN_SIZE:]

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape},  y={y_test.shape}")

## 2. SimpleRNN Model

In [ ]:
def build_simple_rnn(n_steps, n_features=1):
    model = keras.Sequential([
        keras.layers.SimpleRNN(64, return_sequences=True,
                               input_shape=(n_steps, n_features)),
        keras.layers.SimpleRNN(32),
        keras.layers.Dense(1)
    ], name="SimpleRNN")
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

rnn_model = build_simple_rnn(N_STEPS)
rnn_model.summary()

In [ ]:
log_dir = f"../logs/fit/rnn_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
tb_cb   = keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
es_cb   = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

rnn_history = rnn_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    callbacks=[tb_cb, es_cb],
    verbose=1
)

## 3. LSTM Model

In [ ]:
def build_lstm(n_steps, n_features=1):
    model = keras.Sequential([
        keras.layers.LSTM(64, return_sequences=True,
                          input_shape=(n_steps, n_features)),
        keras.layers.LSTM(32),
        keras.layers.Dense(1)
    ], name="LSTM")
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

lstm_model = build_lstm(N_STEPS)
lstm_model.summary()

In [ ]:
log_dir = f"../logs/fit/lstm_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
tb_cb   = keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
es_cb   = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

lstm_history = lstm_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    callbacks=[tb_cb, es_cb],
    verbose=1
)

## 4. GRU Model

In [ ]:
def build_gru(n_steps, n_features=1):
    model = keras.Sequential([
        keras.layers.GRU(64, return_sequences=True,
                         input_shape=(n_steps, n_features)),
        keras.layers.GRU(32),
        keras.layers.Dense(1)
    ], name="GRU")
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

gru_model = build_gru(N_STEPS)
gru_model.summary()

In [ ]:
log_dir = f"../logs/fit/gru_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
tb_cb   = keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
es_cb   = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

gru_history = gru_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    callbacks=[tb_cb, es_cb],
    verbose=1
)

## 5. Training Loss Comparison

In [ ]:
def plot_history(histories, labels):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for history, label in zip(histories, labels):
        axes[0].plot(history.history["loss"],        label=f"{label} train")
        axes[0].plot(history.history["val_loss"], "--", label=f"{label} val")
        axes[1].plot(history.history["mae"],          label=f"{label} train")
        axes[1].plot(history.history["val_mae"],  "--", label=f"{label} val")
    for ax, title in zip(axes, ["MSE Loss", "MAE"]):
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

plot_history(
    [rnn_history, lstm_history, gru_history],
    ["SimpleRNN", "LSTM", "GRU"]
)

## 6. Predictions on Test Set

In [ ]:
rnn_pred  = rnn_model.predict(X_test).flatten()
lstm_pred = lstm_model.predict(X_test).flatten()
gru_pred  = gru_model.predict(X_test).flatten()

plt.figure(figsize=(14, 4))
plt.plot(y_test,    label="Ground truth",  linewidth=1.5, color="black")
plt.plot(rnn_pred,  label="SimpleRNN",     linewidth=0.9, alpha=0.8)
plt.plot(lstm_pred, label="LSTM",          linewidth=0.9, alpha=0.8)
plt.plot(gru_pred,  label="GRU",           linewidth=0.9, alpha=0.8)
plt.title("Test Set Predictions")
plt.xlabel("Time step")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Evaluation Metrics

In [ ]:
models = {
    "SimpleRNN": rnn_model,
    "LSTM":      lstm_model,
    "GRU":       gru_model,
}

print(f"{'Model':<12}  {'Test MSE':>10}  {'Test MAE':>10}")
print("-" * 38)
for name, model in models.items():
    mse, mae = model.evaluate(X_test, y_test, verbose=0)
    print(f"{name:<12}  {mse:>10.5f}  {mae:>10.5f}")

## 8. Multi-step Forecasting

Iteratively feed predictions back as input to forecast multiple steps ahead.

In [ ]:
def multi_step_forecast(model, seed_window, n_ahead):
    """Forecast n_ahead steps by feeding each prediction back as input."""
    window = seed_window.copy()   # shape: (n_steps, 1)
    preds  = []
    for _ in range(n_ahead):
        x    = window[np.newaxis]  # (1, n_steps, 1)
        pred = model.predict(x, verbose=0)[0, 0]
        preds.append(pred)
        window = np.append(window[1:], [[pred]], axis=0)
    return np.array(preds)

N_AHEAD   = 100
seed      = X_test[0]                           # first test window
true_vals = y_all[TRAIN_SIZE : TRAIN_SIZE + N_AHEAD]

lstm_forecast = multi_step_forecast(lstm_model, seed, N_AHEAD)
gru_forecast  = multi_step_forecast(gru_model,  seed, N_AHEAD)

plt.figure(figsize=(12, 4))
plt.plot(true_vals,      label="Ground truth", linewidth=1.5, color="black")
plt.plot(lstm_forecast,  label="LSTM forecast",  linewidth=1.0)
plt.plot(gru_forecast,   label="GRU forecast",   linewidth=1.0)
plt.title(f"{N_AHEAD}-step Ahead Forecast")
plt.xlabel("Steps ahead")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()

## Summary

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| **SimpleRNN** | Lightweight, fast to train | Vanishing gradient on long sequences |
| **LSTM** | Handles long-range dependencies via cell/hidden state gates | More parameters, slower |
| **GRU** | Fewer parameters than LSTM, similar accuracy | Slightly less expressive than LSTM |

For most practical time series tasks, **LSTM** or **GRU** are recommended over `SimpleRNN`.